# DL · Encuentro 2 · Imágenes, CNN y transferencia

De los píxeles a la representación

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [Tres preguntas distintas para una imagen](#scrollTo=tema-dl-vision)
2. [Aprender patrones locales de píxeles](#scrollTo=tema-dl-cnn)
3. [Reutilizar lo aprendido](#scrollTo=tema-dl-transferencia)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · Distinguir etiqueta, cajas y regiones

**Lo que vas a poder hacer:** Elegir entre etiqueta, caja y máscara según la pregunta que debe responder una imagen.

### Entiende la idea

Una imagen digital guarda números en posiciones llamadas píxeles. En RGB (Red, Green, Blue: rojo, verde y azul), cada píxel guarda tres componentes. La visión por computador transforma estos números para responder tareas. Clasificar describe la imagen; detectar localiza objetos; segmentar marca píxeles. Cada tarea necesita respuestas de referencia y una evaluación apropiadas.

**Píxel y canal:** Un píxel es una posición de imagen; un canal guarda una componente, como rojo, verde o azul.

**Caja:** Rectángulo que localiza un objeto y puede incluir fondo a su alrededor.

**Máscara:** Marca de píxeles de interés. Las máscaras de instancias permiten separar objetos de la misma categoría.

### Un caso pequeño antes del programa

**¿Qué respuesta necesita la persona que revisa esta foto?**

Conservamos una escena ilustrada de tres frutos. Cambiamos el encargo: seleccionar la foto, localizar los frutos o delimitar sus regiones visibles.

**Cómo leerlo:** La etiqueta responde para toda la imagen. Las cajas señalan dónde mirar. Las máscaras marcan regiones de píxeles. Son salidas dibujadas para comparar tareas, no resultados de una red.

**Para recordar:** Primero define la pregunta y la salida necesaria. Una etiqueta global no sustituye la localización o los contornos.

### Ahora identifica los datos de esta práctica

Reconocer qué información aporta cada tipo de salida visual.

1. En dl-encuentro-2.ipynb, Tema 1, ejecuta su preparación y el dibujo de tareas visuales.
2. Compara etiqueta de imagen, cajas y máscara binaria para los dos objetos del cuaderno.
3. Explica qué pregunta responde cada salida y qué información todavía no proporciona.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta las celdas en orden; no requiere los temas anteriores. Las celdas nuevas de cuentas pequeñas incluyen sus propias importaciones y datos.

**Qué cambiar y qué conservar:** En el cuaderno hay dos objetos ilustrados y en el aula, tres frutos. Compara el tipo de salida, no esperes que ambos dibujos tengan el mismo conteo.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### Comparar etiqueta, cajas y regiones

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Ilustración exacta de tres TIPOS de salida, usando objetos sintéticos.
from matplotlib.patches import Rectangle
h,w=60,90;fila,col=np.mgrid[:h,:w]
mascara=((fila-30)**2+(col-25)**2<13**2)|((fila-28)**2+(col-65)**2<10**2)
imagen=np.ones((h,w,3))*.93;imagen[mascara]=[.2,.65,.4]
fig,axs=plt.subplots(1,3,figsize=(11,3))
for ax in axs:ax.axis('off')
axs[0].imshow(imagen);axs[0].set_title('Clasificación: «hay objetos»')
axs[1].imshow(imagen);axs[1].add_patch(Rectangle((12,17),26,26,fill=False,ec='red',lw=2));axs[1].add_patch(Rectangle((55,18),20,20,fill=False,ec='red',lw=2));axs[1].set_title('Detección: ¿dónde están?')
axs[2].imshow(mascara,cmap='Greens');axs[2].set_title('Segmentación: ¿qué píxeles?');mostrar('10-tareas-vision')
print('Estas cajas y máscaras se dibujaron a partir de la geometría: no ejecutan YOLO ni U-Net.')


### Después de ejecutar: ¿qué encargo requiere cada salida?

Para seleccionar imágenes basta una etiqueta global. Para señalar dónde está cada objeto hacen falta cajas. Para delimitar su región se usan píxeles marcados. Aquí hay dos objetos dibujados; el aula usa tres frutos. Cambia la escena, no la diferencia entre las tareas.


### Lee, comprueba y explica

**Qué debes poder comprobar:** El cuaderno dibuja respuestas para enseñar tareas; no ejecuta aquí un detector. Su máscara binaria separa objeto y fondo, pero no identifica por separado cada instancia.

**Prueba pequeña:** En el cuaderno hay dos objetos ilustrados y en el aula, tres frutos. Compara el tipo de salida, no esperes que ambos dibujos tengan el mismo conteo.

**Pregunta:** Si necesitas medir la región visible de cada fruto, ¿basta una etiqueta que diga «hay frutos»?

**Respuesta razonada:** No. Necesitas delimitar sus píxeles y separar los objetos, por ejemplo con máscaras de instancias. Convertir esa región a unidades físicas exige además una escala de imagen conocida.

**Registra en tu copia:** qué cambiaste, qué observaste y por qué ocurrió. No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-vision)

[Siguiente tema: Aprender patrones locales de píxeles](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-cnn)


## Tema 2 · Reconocer dígitos y leer un filtro

**Lo que vas a poder hacer:** Seguir cómo una ventana de píxeles produce una celda y cómo varias celdas forman un mapa.

### Entiende la idea

Una CNN (Convolutional Neural Network, red neuronal convolucional) aprende filtros para buscar patrones locales. Un filtro es una pequeña matriz de pesos: se combina con una ventana de píxeles mediante productos y una suma. Repetirlo en distintas posiciones forma un mapa de respuestas. Aquí fijamos el filtro para entender la operación antes de entrenar.

**Ventana:** Región de píxeles que se usa para calcular una respuesta local.

**Filtro o kernel:** Pesos compartidos que se aplican en distintas posiciones. En esta calculadora son dados; una CNN puede aprenderlos.

**Mapa de respuestas:** Cuadrícula de resultados: cada celda corresponde a una posición de la ventana.

### Un caso pequeño antes del programa

**¿Cómo encuentra este filtro un cambio de oscuro a claro?**

La imagen inventada tiene 5×5 píxeles de un canal: 0 oscuro y 1 claro. Movemos un filtro de 3×3, de una posición en una posición y sin añadir borde.

**Cómo leerlo:** La ventana resaltada produce una sola celda del mapa. En la primera posición, los productos suman 3; en una zona uniforme se compensan y dan 0.

**Para recordar:** Una respuesta alta indica un patrón del filtro, no un objeto contado. Entrenar una CNN ajusta filtros a partir de ejemplos.

### Ahora identifica los datos de esta práctica

Distinguir una CNN que aprende de un filtro fijo que ilustra una operación.

1. En dl-encuentro-2.ipynb, Tema 2, ejecuta la preparación y la CNN de imágenes de dígitos.
2. Identifica imágenes de entrada, filtros aprendidos y puntuaciones de las diez clases; lee la evaluación que imprime.
3. Inspecciona el filtro fijo mostrado después y explica por qué sus valores y su mapa son otro ejemplo.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta las celdas en orden; no requiere los temas anteriores. Las celdas nuevas de cuentas pequeñas incluyen sus propias importaciones y datos.

**Qué cambiar y qué conservar:** Una respuesta del filtro no es un objeto detectado. Relaciona cada celda del mapa con la zona de imagen que la produjo.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### DL2: redes e imágenes

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **CNN** — CNN: Convolutional Neural Network (Red neuronal convolucional). Red que usa filtros repetidos sobre pequeñas zonas de una entrada. Los filtros combinan números cercanos y producen mapas de respuestas. La CNN del cuaderno aprende con imágenes de dígitos y sus respuestas conocidas; CNN no significa por sí sola detector de objetos. [Fuente](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)

- **RGB** — RGB: Red, Green, Blue (Rojo, verde y azul). Forma de representar un color mediante tres componentes o canales. Cada canal indica cuánto rojo, verde o azul participa en el color del píxel. La escala depende del formato: en imágenes de ocho bits suele ser 0–255 por canal; una preparación puede convertirla a 0–1. [Fuente](https://www.w3.org/TR/css-color-4/#rgb-functions)


### De los cuadritos a la respuesta

La CNN recibe una imagen de 8×8 y aplica ocho filtros de 3×3 cuyos pesos aprende. Producen ocho mapas de respuestas. ReLU cambia a cero las respuestas negativas. MaxPool toma el máximo de cada zona y reduce los mapas de 8×8 a 4×4. Al final, la red calcula diez puntuaciones, una por dígito, y elige la mayor. Esas puntuaciones se llaman logits, no son diez objetos.

**Cómo leer las figuras:** compara «Real» y «pred.» para cada imagen. Después observa el mapa de un filtro fijo que compara izquierda y derecha; es distinto de los ocho filtros aprendidos y del filtro del aula que compara filas. Sus colores representan respuestas numéricas, no clases de dígitos ni probabilidades.

**Qué probar:** cambia solo EPOCAS de 12 a 1 y vuelve a ejecutar la celda entera. Compara respuestas y pérdida sin prometer que cada imagen empeorará. Si eliges una versión mirando repetidamente la misma reserva, la estás usando como validación. Para reutilizar pesos, continúa en el **Tema 3 de este mismo cuaderno**: carga otro modelo para otra tarea; no convierte esta CNN de dígitos en detector.


### Reconocer dígitos con una CNN

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
from torch.utils.data import TensorDataset,DataLoader
D=load_digits();X=D.images.astype('float32')/16;y=D.target
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEMILLA)
xt=torch.tensor(Xtr[:,None]);yt=torch.tensor(ytr,dtype=torch.long);xe=torch.tensor(Xte[:,None])
torch.manual_seed(SEMILLA)
cnn=nn.Sequential(nn.Conv2d(1,8,kernel_size=3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten(),nn.Linear(8*4*4,10))
opt=torch.optim.Adam(cnn.parameters(),lr=.01);perdida=nn.CrossEntropyLoss();lotes=DataLoader(TensorDataset(xt,yt),batch_size=64,shuffle=True)
EPOCAS=12;hist=[]
for epoca in range(EPOCAS):
    cnn.train();suma=0
    for bx,by in lotes:
        opt.zero_grad();error=perdida(cnn(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    hist.append(suma/len(xt))
cnn.eval()
with torch.no_grad():pred=cnn(xe).argmax(1).numpy()
print('CNN · accuracy en prueba:',round(accuracy_score(yte,pred),3))
fig,axs=plt.subplots(2,6,figsize=(11,4))
for i,ax in enumerate(axs.ravel()):
    ax.imshow(Xte[i],cmap='gray');ax.set_title(f'Real {yte[i]} → pred. {pred[i]}',fontsize=9);ax.axis('off')
mostrar('10-cnn')
# Convolución con filtro fijo para visualizar la operación (no es un filtro aprendido).
filtro=torch.tensor([[[[-1.,0.,1.],[-1.,0.,1.],[-1.,0.,1.]]]])
with torch.no_grad():bordes=torch.nn.functional.conv2d(xe[:1],filtro,padding=1).numpy()[0,0]
fig,axs=plt.subplots(1,3,figsize=(10,3));axs[0].imshow(Xte[0],cmap='gray');axs[0].set_title('Entrada: 8×8 píxeles');axs[1].imshow(bordes,cmap='coolwarm');axs[1].set_title('Respuesta del filtro: cambios horizontales');axs[2].plot(hist,color=AZUL);axs[2].set(title='Error de la CNN',xlabel='Época',ylabel='Pérdida');mostrar('10-convolucion')


### Después de ejecutar: localiza un acierto y un error

Mira una imagen y compara la etiqueta Real con pred. Si no hay errores en las doce imágenes dibujadas, no concluyas que no los hay en toda la prueba: el dibujo solo muestra una parte. El mapa de borde corresponde a otro filtro fijo; sus colores no son clases de dígitos.


### Lee, comprueba y explica

**Qué debes poder comprobar:** La CNN aprende a clasificar dígitos de 8×8; la calculadora usa una imagen inventada de 5×5. No esperas mapas iguales ni atribuyes al filtro fijo la evaluación de la red.

**Prueba pequeña:** Una respuesta del filtro no es un objeto detectado. Relaciona cada celda del mapa con la zona de imagen que la produjo.

**Pregunta:** ¿Mover un filtro elegido a mano significa que ya entrenaste una CNN?

**Respuesta razonada:** No. Calculaste respuestas con pesos fijos. Entrenar requiere una tarea, ejemplos, una pérdida y actualizaciones de parámetros.

**Registra en tu copia:** qué cambiaste, qué observaste y por qué ocurrió. No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-cnn)

[Siguiente tema: Reutilizar lo aprendido](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-transferencia)


## Tema 3 · Reutilizar pesos y revisar categorías

**Lo que vas a poder hacer:** Distinguir usar un modelo previo, entrenar una cabeza nueva y hacer ajuste fino.

### Entiende la idea

Un modelo preentrenado ya aprendió parámetros con datos anteriores. Usarlo tal como está para predecir es inferencia. La transferencia aprovecha representaciones previas para otra tarea: puede entrenar una nueva salida manteniendo fijo el extractor. El ajuste fino modifica también parámetros del extractor. Ambas adaptaciones necesitan ejemplos adecuados y una comprobación independiente.

**Preentrenado:** Modelo con parámetros ajustados previamente a otra colección de ejemplos.

**Extractor y cabeza:** El extractor construye representaciones; la cabeza las convierte en la salida requerida.

**Congelar y ajustar:** Congelar mantiene parámetros fijos. Ajustar entrena parámetros seleccionados; el ajuste fino modifica también el extractor previo.

### Un caso pequeño antes del programa

**¿Puede un detector aprender una categoría nueva cambiando su nombre?**

Un equipo quiere inspeccionar café defectuoso y recibe un detector de objetos comunes. Antes de usarlo, comprueba qué categorías aprendió.

**Cómo leerlo:** Cargar pesos permite inferir. Traducir una etiqueta mejora la lectura. Entrenar una salida nueva o ajustar el extractor son otras acciones: cambian parámetros y requieren datos.

**Para recordar:** Reutilizar una red no demuestra que resuelva una tarea nueva. Hay que comprobar categorías, condiciones y resultados.

### Ahora identifica los datos de esta práctica

Comprobar qué reconoce el detector cargado y qué reutiliza al predecir.

1. En dl-encuentro-2.ipynb, Tema 3, ejecuta las celdas de preparación y carga del modelo.
2. Consulta modelo.names y sigue una categoría disponible hasta una propuesta de la foto.
3. Aplica el filtro de categoría y comprueba que conserva una clase existente sin entrenar pesos.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta las celdas en orden; no requiere los temas anteriores. Las celdas nuevas de cuentas pequeñas incluyen sus propias importaciones y datos.

**Qué cambiar y qué conservar:** Filtrar «persona» conserva una clase existente. Renombrarla como «café» solo cambiaría el texto y sería una interpretación incorrecta.


### Nombres del modelo, los datos y el dispositivo

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **YOLO** — YOLO: You Only Look Once (Solo se mira una vez). Nombre de una familia de modelos de detección que propone categorías y cajas a partir de imágenes. El nombre nació con un enfoque que calcula esas propuestas en una evaluación de la red. Las variantes tienen arquitecturas y procesamiento propios; una caja conservada no demuestra que la propuesta sea correcta. [Fuente](https://arxiv.org/abs/1506.02640)

- **COCO** — COCO: Common Objects in Context (Objetos comunes en contexto). Conjunto de imágenes con anotaciones de objetos en escenas cotidianas. Sirve para desarrollar y comparar modelos de visión; no es un detector por sí mismo. El modelo usado en este cuaderno ofrece 80 categorías de ese conjunto, no cualquier objeto ni cualquier defecto agrícola. [Fuente](https://arxiv.org/abs/1405.0312)

- **CPU** — CPU: Central Processing Unit (Unidad central de procesamiento). Procesador que ejecuta las instrucciones generales de un programa. En los cuadernos, cpu identifica el dispositivo que realiza los cálculos. No es una categoría aprendida ni una medida de calidad. [Fuente](https://www.intel.com/content/www/us/en/products/docs/processors/cpu-vs-gpu.html)

- **RGB** — RGB: Red, Green, Blue (Rojo, verde y azul). Forma de representar un color mediante tres componentes o canales. Cada canal indica cuánto rojo, verde o azul participa en el color del píxel. La escala depende del formato: en imágenes de ocho bits suele ser 0–255 por canal; una preparación puede convertirla a 0–1. [Fuente](https://www.w3.org/TR/css-color-4/#rgb-functions)


In [ ]:
import sys, subprocess
from importlib import metadata

VERSION_ULTRALYTICS = '8.4.175'
try:
    version_instalada = metadata.version('ultralytics')
except metadata.PackageNotFoundError:
    version_instalada = None
if version_instalada != VERSION_ULTRALYTICS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'ultralytics=={VERSION_ULTRALYTICS}'])
import ultralytics
if ultralytics.__version__ != VERSION_ULTRALYTICS:
    raise RuntimeError('Ultralytics sigue cargado con otra versión. Reinicia el entorno y ejecuta de nuevo desde la preparación.')
print('Ultralytics:', ultralytics.__version__)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, FileLink
import torch
from ultralytics import YOLO
from ultralytics.utils import ASSETS

SALIDA = Path('resultados_yolo')
SALIDA.mkdir(exist_ok=True)
torch.set_num_threads(2)
TRADUCCION = {'person':'persona','bus':'bus','car':'automóvil','truck':'camión',
              'bicycle':'bicicleta','motorcycle':'motocicleta','chair':'silla',
              'bottle':'botella','cup':'taza','laptop':'portátil','cell phone':'teléfono',
              'book':'libro','backpack':'mochila','tie':'corbata'}
def nombre_es(nombre):
    return TRADUCCION.get(nombre, nombre)
print('PyTorch:',torch.__version__)
print('Los nombres no traducidos conservan la categoría original en inglés.')


In [ ]:
modelo = YOLO('yolo26n.pt')
categorias = pd.DataFrame([{'id':i,'clase_original':nombre,'nombre':nombre_es(nombre)}
                           for i,nombre in modelo.names.items()])
print('Categorías conocidas:',len(categorias))
print('Ejemplos del catálogo:',', '.join(categorias.nombre.head(15)))
print('Son solo ejemplos; una categoría no mostrada aún puede existir. La búsqueda consulta el catálogo completo.')
BUSCAR_CATEGORIA='café' # después prueba 'persona' sin cambiar los pesos
consulta=BUSCAR_CATEGORIA.strip().casefold()
coincidencias=categorias[(categorias.nombre.str.casefold().str.contains(consulta,regex=False))|
                         (categorias.clase_original.str.casefold().str.contains(consulta,regex=False))] if consulta else categorias.iloc[:0]
if coincidencias.empty:
    print('No hay coincidencias para',repr(BUSCAR_CATEGORIA),'en este catálogo. No renombres otra clase para fingir que la reconoce.')
else:
    print('Coincidencias en todo el catálogo:',', '.join(coincidencias.nombre))


In [ ]:
foto = ASSETS / 'bus.jpg'
original = Image.open(foto).convert('RGB')
print('Ancho y alto:',original.size,'· canales: RGB')
fig,ax=plt.subplots(figsize=(5,7))
ax.imshow(original);ax.axis('off');ax.set_title('Entrada: la foto sin predicciones')
plt.tight_layout();plt.savefig(SALIDA/'01_original.png',dpi=130);plt.show()


In [ ]:
UMBRAL = 0.25
resultado = modelo.predict(source=str(foto), conf=UMBRAL, imgsz=640,
                           device='cpu', verbose=False)[0]
print('Detecciones conservadas:',len(resultado.boxes))
resultado.save(filename=str(SALIDA/'02_foto_detectada.jpg'))
fig,axs=plt.subplots(1,2,figsize=(10,7))
axs[0].imshow(original);axs[0].set_title('Antes: píxeles')
axs[1].imshow(resultado.plot()[:,:,::-1]);axs[1].set_title('Después: categorías, cajas y puntuaciones')
for ax in axs:ax.axis('off')
plt.tight_layout();plt.savefig(SALIDA/'03_antes_despues.png',dpi=130);plt.show()


In [ ]:
def tabla_detecciones(r):
    columnas=['deteccion','clase_id','clase_original','categoria','puntuacion',
              'x1','y1','x2','y2','ancho_px','alto_px']
    filas=[]
    for i,caja in enumerate(r.boxes, start=1):
        clase=int(caja.cls.item());nombre=r.names[clase]
        x1,y1,x2,y2=caja.xyxy[0].cpu().numpy().tolist()
        filas.append([i,clase,nombre,nombre_es(nombre),float(caja.conf.item()),
                      x1,y1,x2,y2,x2-x1,y2-y1])
    return pd.DataFrame(filas,columns=columnas)

tabla = tabla_detecciones(resultado)
display(tabla.round(3))
assert len(tabla)==len(resultado.boxes)
if not tabla.empty:
    primera=tabla.iloc[0]
    print('Primera detección:',primera.categoria,
          '· puntuación:',round(primera.puntuacion,3),
          '· tamaño:',round(primera.ancho_px),'×',round(primera.alto_px),'px')


In [ ]:
id_persona=next(i for i,n in modelo.names.items() if n=='person')
solo_personas=modelo.predict(str(foto),classes=[id_persona],conf=UMBRAL,
                             imgsz=640,device='cpu',verbose=False)[0]
display(tabla_detecciones(solo_personas).round(3))
if len(solo_personas.boxes):
    x1,y1,x2,y2=solo_personas.boxes.xyxy[0].cpu().numpy()
    recorte=original.crop((max(0,int(x1)),max(0,int(y1)),
                           min(original.width,int(x2)),min(original.height,int(y2))))
    fig,ax=plt.subplots(figsize=(3,4));ax.imshow(recorte);ax.axis('off')
    ax.set_title('Un recorte de la primera caja de persona');plt.tight_layout();plt.show()
else:print('No hay personas conservadas con este umbral.')


### Lee, comprueba y explica

**Qué debes poder comprobar:** Puedes explicar por qué el cuaderno ejecuta inferencia y no ajuste fino. Que una clase esté disponible tampoco garantiza buen resultado en cualquier foto local.

**Prueba pequeña:** Filtrar «persona» conserva una clase existente. Renombrarla como «café» solo cambiaría el texto y sería una interpretación incorrecta.

**Pregunta:** Si dejas fijo el extractor y entrenas solo una cabeza nueva, ¿hiciste ajuste fino del extractor?

**Respuesta razonada:** No. Hiciste transferencia mediante una nueva cabeza. El ajuste fino cambia además parte o todo el extractor. El cuaderno de este tema solo realiza inferencia.

**Registra en tu copia:** qué cambiaste, qué observaste y por qué ocurrió. No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-transferencia)

[Siguiente tema: RNN y LSTM: el orden puede cambiar el significado](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-secuencias)


## Cierra el encuentro con una explicación

1. **Tres preguntas distintas para una imagen:** Primero define la pregunta y la salida necesaria. Una etiqueta global no sustituye la localización o los contornos.
2. **Aprender patrones locales de píxeles:** Una respuesta alta indica un patrón del filtro, no un objeto contado. Entrenar una CNN ajusta filtros a partir de ejemplos.
3. **Reutilizar lo aprendido:** Reutilizar una red no demuestra que resuelva una tarea nueva. Hay que comprobar categorías, condiciones y resultados.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
